# Обратная нормализация текста

Финальное решение задачи BIO-разметки русских, казахских и смешанных предложений. Модель распознаёт числовые выражения, даты, время, величины, email и специальные словарные классы.

**Сохранённый leaderboard span-level F1: приблизительно 0.914.**


## Запуск

Положите `train.csv` и `test.csv` рядом с ноутбуком и выполните **Run All**. По умолчанию сохранены исходные размеры батчей, использованные в финальном решении. При нехватке памяти задайте переменные `ITN_TRAIN_BATCH_SIZE` и `ITN_INFERENCE_BATCH_SIZE`.

Ноутбук расширяет словарь RuBERT, выполняет основное обучение и быструю адаптацию, применяет BIO-Viterbi и сохраняет проверенный `solution.csv`.


In [ ]:
import hashlib
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForTokenClassification, AutoTokenizer

SEED = 123
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LENGTH = 384
TRAIN_BATCH_SIZE = int(os.getenv("ITN_TRAIN_BATCH_SIZE", "192"))
INFERENCE_BATCH_SIZE = int(os.getenv("ITN_INFERENCE_BATCH_SIZE", "256"))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

CLASSES = ["CARDINAL", "ORDINAL", "DECIMAL", "DATE", "TIME", "MEASURE", "EMAIL", "WHITELIST"]
LABELS = ["O"] + [label for name in CLASSES for label in (f"B-{name}", f"I-{name}")]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}


def read_sentences(path: str, with_labels: bool = True):
    frame = pd.read_csv(path, dtype="string")
    sentence_ids = frame["sent_id"].to_numpy()
    starts = np.r_[0, np.flatnonzero(sentence_ids[1:] != sentence_ids[:-1]) + 1]
    stops = np.r_[starts[1:], len(frame)]
    tokens = frame["token"].to_numpy()
    sentences = [tokens[start:stop].tolist() for start, stop in zip(starts, stops)]
    unique_ids = sentence_ids[starts].tolist()
    if not with_labels:
        return unique_ids, sentences, frame
    encoded = frame["label"].map(LABEL_TO_ID).to_numpy()
    targets = [encoded[start:stop].tolist() for start, stop in zip(starts, stops)]
    return unique_ids, sentences, targets, frame


def iter_batches(indices, batch_size, shuffle=True):
    indices = list(indices)
    if shuffle:
        random.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        yield indices[start:start + batch_size]


def encode_batch(tokenizer, sentences, targets, indices):
    encoded = tokenizer(
        [sentences[index] for index in indices],
        is_split_into_words=True,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    token_labels = torch.full(encoded.input_ids.shape, -100)
    word_positions = []
    for row, sentence_index in enumerate(indices):
        previous_word = None
        positions = {}
        for token_position, word_index in enumerate(encoded.word_ids(row)):
            if word_index is not None and word_index != previous_word:
                positions[word_index] = token_position
                if targets is not None:
                    token_labels[row, token_position] = targets[sentence_index][word_index]
            previous_word = word_index
        if len(positions) != len(sentences[sentence_index]):
            raise ValueError(
                f"Sentence {sentence_index} exceeds MAX_LENGTH={MAX_LENGTH}. "
                "Increase MAX_LENGTH or split long sentences before training."
            )
        word_positions.append(positions)
    encoded = {name: value.to(DEVICE) for name, value in encoded.items()}
    return encoded, token_labels.long().to(DEVICE), word_positions


def train_model(model, tokenizer, sentences, targets, indices, learning_rate, epochs=1, limit=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    model.train()
    for epoch in range(epochs):
        batches = iter_batches(indices, TRAIN_BATCH_SIZE)
        for step, batch_indices in enumerate(batches, start=1):
            if limit and step > limit:
                break
            inputs, labels, _ = encode_batch(tokenizer, sentences, targets, batch_indices)
            optimizer.zero_grad()
            logits = model(**inputs).logits
            loss = F.cross_entropy(
                logits.view(-1, len(LABELS)),
                labels.view(-1),
                ignore_index=-100,
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if step % 200 == 0:
                print(f"epoch={epoch + 1} step={step} loss={loss.item():.5f}")


def transition_matrix(targets, indices):
    counts = np.full((len(LABELS) + 1, len(LABELS)), 0.1)
    start_state = len(LABELS)
    for index in indices:
        previous = start_state
        for label in targets[index]:
            counts[previous, label] += 1
            previous = label
    return np.log(counts / counts.sum(axis=1, keepdims=True))


def bio_viterbi(emissions, transitions, alpha=0.2):
    scores = alpha * transitions[len(LABELS)] + emissions[0]
    backpointers = []
    for position in range(1, len(emissions)):
        candidates = scores[:, None] + alpha * transitions[:len(LABELS)]
        best_previous = candidates.argmax(axis=0)
        scores = candidates[best_previous, np.arange(len(LABELS))] + emissions[position]
        backpointers.append(best_previous)

    label = int(scores.argmax())
    sequence = [label]
    for best_previous in reversed(backpointers):
        label = int(best_previous[label])
        sequence.append(label)
    sequence.reverse()

    current_class = None
    for index, label in enumerate(sequence):
        if label == 0:
            current_class = None
            continue
        class_index = (label - 1) // 2
        is_inside = label % 2 == 0
        if is_inside and class_index != current_class:
            sequence[index] = 1 + 2 * class_index
        current_class = class_index
    return sequence


assert Path("train.csv").exists() and Path("test.csv").exists(), "Add train.csv and test.csv"
sentence_ids, train_sentences, train_targets, train_frame = read_sentences("train.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
domain_tokens = {
    token
    for sentence, labels in zip(train_sentences, train_targets)
    for token, label in zip(sentence, labels)
    if label
} - set(tokenizer.get_vocab())
tokenizer.add_tokens(sorted(domain_tokens))

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    ignore_mismatched_sizes=True,
    id2label={index: label for index, label in enumerate(LABELS)},
    label2id=LABEL_TO_ID,
)
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
model.to(DEVICE)

train_indices = [index for index, sid in enumerate(sentence_ids) if int(sid[1:]) % 20]
adapt_indices = [
    index
    for index, sid in enumerate(sentence_ids)
    if not int(sid[1:]) % 20 or int(sid[1:]) % 7 == 1
]

train_model(model, tokenizer, train_sentences, train_targets, train_indices, 1e-4, epochs=2)
train_model(model, tokenizer, train_sentences, train_targets, adapt_indices, 2.5e-5, epochs=1)

MODEL_DIR = Path("itn_model_final")
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

test_ids, test_sentences, test_frame = read_sentences("test.csv", with_labels=False)
transitions = transition_matrix(train_targets, range(len(train_targets)))
model.eval()
predictions = []

with torch.no_grad():
    batches = iter_batches(range(len(test_sentences)), INFERENCE_BATCH_SIZE, shuffle=False)
    for step, batch_indices in enumerate(batches):
        inputs, _, positions = encode_batch(tokenizer, test_sentences, None, batch_indices)
        logits = model(**inputs).logits.float().cpu().numpy()
        for row, sentence_index in enumerate(batch_indices):
            emissions = np.stack([
                logits[row, positions[row][word_index]]
                for word_index in range(len(test_sentences[sentence_index]))
            ])
            predictions.append(bio_viterbi(emissions, transitions))
        if step % 100 == 0:
            print(f"inference batch={step}")

submission = test_frame[["sent_id", "token_id"]].copy()
submission["label"] = [LABELS[label] for sentence in predictions for label in sentence]
allowed_labels = {"O"} | {f"{prefix}-{name}" for name in CLASSES for prefix in ("B", "I")}

assert len(submission) == len(test_frame)
assert not submission.duplicated(["sent_id", "token_id"]).any()
assert not submission["label"].isna().any()
assert submission["label"].isin(allowed_labels).all()

submission.to_csv("solution.csv", index=False)
checksum = hashlib.sha256(Path("solution.csv").read_bytes()).hexdigest()
print(f"Saved solution.csv: {len(submission):,} rows, sha256={checksum}")
